# S9b — sample-image figures for Paper 1 (CPU only, ≈ 5 min, no GPU quota)
Shows real test mammograms next to the S8 **held-out test** generated images (no new training or generation).
No cherry-picking: the first 3 generated images per class; real images drawn with a fixed seed. All generated rows share
the same test noise bank, so column *j* of every generated row starts from the **same** initial noise.

**Inputs (Add Input → Your Work):** the **S8 replication** notebook output (contains `S8/`, `cmmd_data/`, `clean_data/`).
**Accelerator: None (CPU)** · Internet not needed · Save & Run All (Commit). Download `S9b_sample_figures.zip`.

In [ ]:
# ===================== CELL 1 — code + inputs =====================
import os, sys, glob, json, shutil, subprocess
from pathlib import Path
OUT = "/kaggle/working/S9b"; WD = "/kaggle/working/s9b_code"
os.makedirs(OUT, exist_ok=True); os.makedirs(WD, exist_ok=True)
Path(f"{WD}/s9b_samples.py").write_text(r'''# s9b_samples.py - sample-image figures for Paper 1 (CPU only, no training, no new generation).
# Uses ONLY images that already exist: the S8 held-out TEST generated sets and the real test caches.
# No cherry-picking: the first K images of each class are shown (generated) and K images per class drawn with a fixed
# seed (real). Because all arms share the same test noise bank, column j of every generated row starts from the SAME
# initial noise, so differences between rows are caused by the model, not by the noise.
# usage: python s9b_samples.py <S8_dir> <cmmd_data_dir> <clean_data_dir> <out_dir> [K]
import os, sys, json
import numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

S8, DC, DB, OUT = sys.argv[1:5]
K = int(sys.argv[5]) if len(sys.argv) > 5 else 3
os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "savefig.dpi": 300, "savefig.bbox": "tight"})

PANELS = {
    "CMMD": {"real": os.path.join(DC, "test_cache.npz"),
             "rows": [("tc40", "Baseline (epoch 40)"), ("tcm0", "λ = 0 control (C1)"), ("tcfp", "FP residual (C1)"),
                      ("tc40_nl", "Acquisition-noise layer (C4)")]},
    "CBIS-DDSM": {"real": os.path.join(DB, "test_cache.npz"),
                  "rows": [("tbm0", "λ = 0 control, epoch 92 (C2)"), ("tbfp", "FP residual, epoch 92 (C2)"),
                           ("tsm0", "λ = 0 control (C5)"), ("tsnlf", "NLF loss (C5)")]},
}
LAB = {0: "benign", 1: "malignant"}
info = {"K_per_class": K, "rule": "generated: first K images of each class (shared test noise bank); real: K per class, rng seed 0"}

def load(path):
    d = np.load(path)
    return d["x"], d["y"].astype(int)

def pick_gen(y):
    return np.concatenate([np.where(y == c)[0][:K] for c in (0, 1)])

def pick_real(y):
    rng = np.random.default_rng(0)
    return np.concatenate([np.sort(rng.choice(np.where(y == c)[0], K, replace=False)) for c in (0, 1)])

def show(ax, img):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255, interpolation="lanczos")
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)

for ds, P in PANELS.items():
    xr, yr = load(P["real"]); ir = pick_real(yr)
    rows = [("Real (test set)", xr[ir], yr[ir])]
    used = {"real": {"file": os.path.basename(P["real"]), "indices": ir.tolist()}}
    for name, title in P["rows"]:
        f = os.path.join(S8, f"gen_{name}.npz")
        if not os.path.exists(f):
            print(f"!! {f} missing - row skipped"); continue
        xg, yg = load(f); ig = pick_gen(yg)
        rows.append((title, xg[ig], yg[ig])); used[name] = {"file": os.path.basename(f), "indices": ig.tolist(), "n_total": int(len(yg))}
    info[ds] = used
    n = 2 * K
    # ---------- full images ----------
    fig, axs = plt.subplots(len(rows), n, figsize=(1.45 * n + 1.9, 1.5 * len(rows)), squeeze=False)
    for r, (title, X, Y) in enumerate(rows):
        for c in range(n):
            show(axs[r, c], X[c])
            if r == 0:
                axs[r, c].set_title(LAB[int(Y[c])], fontsize=8)
        axs[r, 0].set_ylabel(title, rotation=0, ha="right", va="center", fontsize=8.5, labelpad=6)
    fig.subplots_adjust(wspace=0.04, hspace=0.06)
    tag = "cmmd" if ds == "CMMD" else "cbis"
    fig.savefig(os.path.join(OUT, f"fig_samples_{tag}.png")); plt.close(fig)
    # ---------- 64 x 64 centre-of-breast crops (texture), same images ----------
    fig, axs = plt.subplots(len(rows), n, figsize=(1.45 * n + 1.9, 1.5 * len(rows)), squeeze=False)
    for r, (title, X, Y) in enumerate(rows):
        for c in range(n):
            img = X[c].astype(float)
            m = img > 0.04 * 255                                   # breast mask, same threshold as the saturation metric
            ys, xs = np.nonzero(m) if m.any() else (np.array([128]), np.array([128]))
            cy, cx = int(np.median(ys)), int(np.median(xs))
            cy, cx = min(max(cy, 32), 224), min(max(cx, 32), 224)
            show(axs[r, c], X[c][cy - 32:cy + 32, cx - 32:cx + 32])
            if r == 0:
                axs[r, c].set_title(LAB[int(Y[c])], fontsize=8)
        axs[r, 0].set_ylabel(title, rotation=0, ha="right", va="center", fontsize=8.5, labelpad=6)
    fig.subplots_adjust(wspace=0.04, hspace=0.06)
    fig.savefig(os.path.join(OUT, f"fig_samples_{tag}_crops.png")); plt.close(fig)
    print(f"{ds}: {len(rows)} rows x {n} images -> fig_samples_{tag}.png, fig_samples_{tag}_crops.png")

json.dump(info, open(os.path.join(OUT, "samples_used.json"), "w"), indent=1)
print("samples_used.json written (exact files and indices shown)")
''')
def walk_find(name, root="/kaggle/input"):
    out = []
    for dp, dns, fns in os.walk(root):
        dns[:] = [d for d in dns if d not in ("CMMD", "jpeg", "U24_synthesis", ".git", "ckpt", "train_logs", "logs")]
        if name in fns:
            out.append(os.path.join(dp, name))
    return sorted(out)
def need_dir(marker, parent):
    h = [os.path.dirname(p) for p in walk_find(marker) if os.path.basename(os.path.dirname(p)) == parent]
    if not h:
        raise SystemExit(f"!! {parent}/{marker} not found -> Add Input -> the S8 replication notebook output")
    print(f"{parent:10s} -> {h[0]}"); return h[0]
DATA_CMMD = need_dir("cmmd_summary.json", "cmmd_data"); DATA_CBIS = need_dir("clean_summary.json", "clean_data")
S8_DIR = need_dir("S8_state.json", "S8")
for n in ("tc40", "tcm0", "tcfp", "tc40_nl", "tbm0", "tbfp", "tsm0", "tsnlf"):
    print(f"  {'ok     ' if os.path.exists(f'{S8_DIR}/gen_{n}.npz') else 'MISSING'} gen_{n}.npz")
print("Cell 1 done.")

In [ ]:
# ===================== CELL 2 — make the figures + zip =====================
r = subprocess.run([sys.executable, "s9b_samples.py", S8_DIR, DATA_CMMD, DATA_CBIS, OUT, "3"], cwd=WD, capture_output=True, text=True)
print(r.stdout); print(r.stderr[-3000:] if r.returncode else "")
from IPython.display import Image, display
for f in sorted(glob.glob(f"{OUT}/*.png")):
    print(os.path.basename(f)); display(Image(f, width=900))
z = shutil.make_archive("/kaggle/working/S9b_sample_figures", "zip", OUT)
print("ZIP:", z, f"({os.path.getsize(z) / 1e6:.1f} MB) | return code {r.returncode}")